# Day 55 — Business-impact simulation: reminders as policy

Status: COMPLETE. Flagged-risk appointments get a $1 reminder that averts
30% of would-be no-shows (stated assumption): best policy saves $178k (26%)
vs doing nothing — but barely beats remind-everyone. 15/15 tests pass.

In [1]:
import json
from pathlib import Path

s = json.loads(Path("../models/simulation.json").read_text())
print(f"do-nothing:          ${s['baseline']['total']:,.0f} (4,464 missed x $150)")
by_thr = {r['threshold']: r for r in s['policies'] if isinstance(r['threshold'], float)}
for t in (0.05, 0.15, 0.5):
    r = by_thr[t]
    tag = " (best)" if r == s["best"] else (" (tuned)" if t == 0.15 else "")
    print(f"remind @{t}{tag}: ${r['total']:,.0f} "
          f"({r['prevented']:,} prevented, ${r['sms_spend']:,.0f} SMS)")
re_ = [r for r in s["policies"] if r["threshold"] == "remind-everyone"][0]
print(f"remind EVERYONE:      ${re_['total']:,.0f} ({re_['prevented']:,} prevented, "
      f"${re_['sms_spend']:,.0f} SMS)")
print("plot: docs/simulation.png (cost curve + do-nothing line)")

do-nothing:          $669,600 (4,464 missed x $150)
remind @0.05 (best):  $492,021 (1,330 prevented, $21,921 SMS)
remind @0.15 (tuned): $494,550 (1,294 prevented, $19,050 SMS)
remind @0.5:          $529,451 (1,004 prevented, $10,451 SMS)
remind EVERYONE:      $491,756 (1,333 prevented, $22,106 SMS)
plot: docs/simulation.png (cost curve + do-nothing line)


## Findings: the uncomfortable honest ones

1. **Targeting barely beats blanket reminders** ($492k vs $492k): at $1/SMS
   and 30% effectiveness, the breakeven flag probability is ~2%, so the model
   recommends reminding nearly everyone. The model's value here is *ranking*
   for scarce interventions (calls, overbooking slots), not SMS triage.
2. **Effectiveness runs the whole table**: every dollar above assumes 0.3.
   A pilot must measure it; at 0.1 the savings mostly evaporate. Stated in
   `simulation.json`, not buried.
3. **$178k saved (26%) is the interview number** — with the caveats attached.
   A candidate who quotes savings without assumptions is selling; one who
   attaches them is engineering.